# Simulation de montants de sinistres (attritionnel / extrême)

Deux régimes recollés à un **seuil net**, pour tester si Threshold Studio retrouve le seuil de bascule vers la queue GPD :

| Régime | Loi | Plage de valeurs |
|---|---|---|
| Attritionnel | Lognormale (tronquée) | `[0, U]` |
| Extrême | Pareto généralisée (GPD) | `[U, +inf[` |

**Vérité terrain** : `U` est le seuil au-dessus duquel les sinistres extrêmes suivent *exactement* une GPD — c'est celui que les méthodes EVT de l'app (MRL, Hill, GPD stabilité…) devraient détecter.

Le fichier Excel exporté (colonne `montant_sinistre`) se charge directement dans l'app.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# ── Paramètres (à ajuster librement) ────────────────────────────────────
SEED = 42
N    = 10_000          # nombre total de sinistres

# Seuil de recollement (vérité terrain : attritionnel -> extrême GPD)
U = 50_000

# Proportions de chaque régime (somme = 1)
P_ATTRI, P_EXTREME = 0.97, 0.03

# Attritionnel : lognormale
MU_LN, SIGMA_LN = np.log(1_500), 0.60   # médiane ~1500 €

# Extrême : GPD au-dessus de U
GPD_XI, GPD_BETA = 0.40, 15_000         # xi>0 => queue lourde

assert abs(P_ATTRI + P_EXTREME - 1.0) < 1e-9, "Les proportions doivent sommer à 1"

In [ ]:
# ── Simulation par échantillonnage tronqué (inverse-CDF) ─────────────────
rng = np.random.default_rng(SEED)

# Effectifs de chaque régime
n1, n2 = rng.multinomial(N, [P_ATTRI, P_EXTREME])

# 1. Attritionnel : lognormale tronquée sur [0, U]
body  = stats.lognorm(s=SIGMA_LN, scale=np.exp(MU_LN))
u     = rng.uniform(0.0, body.cdf(U), n1)
attri = body.ppf(u)

# 2. Extrême : excès GPD ajoutés au seuil U
extreme = U + stats.genpareto.rvs(c=GPD_XI, scale=GPD_BETA, size=n2, random_state=rng)

print(f"Attritionnel : {n1:>6}  | extrême : {n2:>5}")

In [ ]:
# ── Assemblage en DataFrame (mélangé) ────────────────────────────────────
montants = np.concatenate([attri, extreme])
regimes  = np.array(["attritionnel"] * n1 + ["extreme"] * n2)

perm = rng.permutation(N)
df = pd.DataFrame({
    "montant_sinistre": np.round(montants[perm], 2),
    "regime": regimes[perm],            # vérité terrain, pour vérification
})
df.head()

In [ ]:
# ── Résumé et seuil attendu ──────────────────────────────────────────────
print(df["montant_sinistre"].describe().round(0))
p_u = (df["montant_sinistre"] <= U).mean() * 100
print(f"\nSeuil vérité terrain :")
print(f"  U = {U:>7,} €  (~P{p_u:.1f})  attritionnel -> extrême  <-- seuil GPD à détecter")

In [ ]:
# ── Visualisation (échelle log) ──────────────────────────────────────────
m = df["montant_sinistre"].values
bins = np.logspace(np.log10(m.min()), np.log10(m.max()), 60)

fig, ax = plt.subplots(figsize=(10, 5))
for reg, color in [("attritionnel", "#4C9BE8"), ("extreme", "#D7484B")]:
    ax.hist(df.loc[df.regime == reg, "montant_sinistre"], bins=bins, color=color, alpha=0.75, label=reg)
ax.axvline(U, color="#000", ls="--", lw=1.2); ax.text(U, ax.get_ylim()[1]*0.9, " U", color="#000")
ax.set_xscale("log"); ax.set_xlabel("Montant du sinistre (€, échelle log)"); ax.set_ylabel("Effectif")
ax.set_title("Sinistres simulés — attritionnel (lognormal) / extrême (GPD)"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Export Excel pour Threshold Studio ───────────────────────────────────
# Nécessite openpyxl :  pip install openpyxl
out = "sinistres_simules.xlsx"
df.to_excel(out, index=False, sheet_name="sinistres")
print(f"Écrit : {out}  ({len(df)} lignes)")
print("Dans l'app : charger ce fichier, colonne cible = 'montant_sinistre', et vérifier que le seuil détecté ≈ U.")